# Restraint Vis

`restraint_vis` is a thin wrapper around `molviewspec` designed for visualizing restraint information form structural models obtained using integrative or hybrid modeling.

The following cell details the input options availible and an explanation of the different style configurations.

## User Input:

### cif file
The path to your cif file. For the time being, this should be a local file.

### filters
We provide some convinience filters to visualize only a subset of the availible restraints. The user should provide a list such that none, one, or multiple filters can be chained together. The following filters are availible:

  - 'across_chains': filter to restraints across chains
  - 'violated': filter to only violated restraints
  - 'compliant': filter to only compliant restraints
  - 'diversity_filter': minimal number of restraints to show one per residue

### max restraints
The maximum number of restraints to visualize. After filters are applied, will randomly downsample to this number of restraints.

### sub_style_mode
When visulazing restraints you may want to style some restraints differently than others within the same visualization. For example, showing the violated restraints in `red` and compliant restraints in `green`. We call these `sub_styles`. These sub_styles inherit all of the parent's styling making it easy to set the global base style and have the sub_styles overwrite only certain parts of the config.

The underlying implementation of the sub_styles is very flexible for the user writing their own functions. However, we have three built-in modes currently implemented:
  - `None`: Don't apply any sub_styles, all restraints have same style
  - 'violated_and_compliant': apply the "violated" sub_style to restraints that are violated and "compliant" to restraints that are compliant
  - 'inter_and_intra_chain': apply "inter" sub_style to restraints that span chains and "intra" sub_style to restraints within the same chain


### Style json
A json file providing user defined styles and sub_styles. To use the defaults, you may set this to `None`. The format is to provide the function name (`visualize_restraint`, or `visualize_macromolecule`) as the top-level key, followed by a `default` child key in which the function's base defaults should be specified. These are dictionaries that are passed directly to `molviewspec`. Sub_styles (described above) can be added as additional children keys to the parent function. Refer to the below example:

```
{"visualize_restraint":
    "default": {
                    "representation_params": {"type": "ball_and_stick"},
                    "color_params"         : {"color": "blue"}
                },

     "violated": {
                    "color_params": {"color": "red"}
                  },

      "compliant": {
                    "representation_params": {"type": "spacefill"}
                   }
}
```

With the above config, a restraint with no specified sub_style will be visualized as a `blue` `ball_and_stick`. If the `violated` sub_style is requested, the restraint will be a `red` `ball_and_stick` (inheriting `ball_and_stick` from the default style). If the `compliant` subs_style is used the restraint will be `blue` `spacefill` (inheriting the `blue` from the default style).

In [1]:
##############
# User Input #
##############

cif_file = "9a3v.cif"
filters = ["across_chains"]
sub_style_mode = "violated_and_compliant"
max_restraints = 50
style_json = "test_user_config1.json"

output_file = "9a3v_inter-chain_restraints.mvsj"

##################################################################
##################################################################
##################################################################

from restraint_vis import core, config
import molviewspec as mvs
from pathlib import Path

# Validate inputs
FILTERS = {
               "across_chains": core.filter_funcs.across_chains,
               "diversity_filter": core.filter_funcs.diversity_filter,
               "violated": core.filter_funcs.violated,
               "compliant": core.filter_funcs.compliant,
           }

def violated_and_compliant_mode(structure, df):
    df = core.filter_funcs.get_solved_distance(df)
    df["sub_style"] = df["compliant"].map({True: "compliant", False: "violated"})

    # Sort so that violated restraints are visualized last, 
    # so that if a residue is part of a compliant AND violated restraint
    # it is colored as violated
    df = df.sort_values("sub_style", ascending=False)
    display(df)
    core.visualize_restraints(structure, df, sub_style_col="sub_style")


def inter_and_intra_chain_mode(structure, df):
    df["sub_style"] = (df["asym_id_1"] == df["asym_id_2"]).map({True: "intra", False: "inter"})

    # Sort so that inter restraints are visualized last, 
    # so that if a residue is part of a inter AND intra restraint
    # it is colored as inter
    df = df.sort_values("sub_style", ascending=False)
    core.visualize_restraints(structure, df, sub_style_col="sub_style")


SUB_STYLE_MODES = {

            "violated_and_compliant": violated_and_compliant_mode,
            "inter_and_intra_chain": inter_and_intra_chain_mode,
}

for filter_ in FILTERS:
    if not filter_ in FILTERS:
        raise ValueError(f"User provided filter ({filter_}) is not recognized. Please choose from {list(FILTERS.keys())}")

if not Path(cif_file).exists():
    raise FileNotFoundError(f"The specified cif file ({cif_file} does not exist!")

if not sub_style_mode in SUB_STYLE_MODES:
    raise ValueError(f"The requested sub_style_mode ({sub_style_mode}) is unknown! Please choose from {list(SUB_STYLE_MODES.keys())}")

if style_json is not None:
    if not Path(style_json).exists():
        raise fileNotFoundError(f"The specified style json file ({style_json}) does not exist!")

    config.set_style_from_json(style_json)

print(f"Will read {cif_file} and visualize restraints filtered by {filters} with style set by {style_json} (will show at most {max_restraints} restraints)")

if sub_style_mode is not None:
    print(f"Using {sub_style_mode} to apply sub_styles to restraints")
    
print(f"Will write output to {output_file}")

Will read 9a3v.cif and visualize restraints filtered by ['across_chains'] with style set by test_user_config1.json (will show at most 50 restraints)
Using violated_and_compliant to apply sub_styles to restraints
Will write output to 9a3v_inter-chain_restraints.mvsj


In [2]:
# Parsing
cif = core.parse_file(cif_file)
restraint_df = core.get_restraints(cif)

print("Restraint table pre-filtering:")
restraint_df

Restraint table pre-filtering:


,entity_id_1,asym_id_1,seq_id_1,comp_id_1,atom_id_1,entity_id_2,asym_id_2,seq_id_2,comp_id_2,atom_id_2,model_granularity,distance_threshold,restraint_type,atom_id_1_coords,atom_id_2_coords
0,1,A,643,GLU,CA,2,B,121,LYS,CA,by-residue,25.0,upper bound,"(23.083, 20.729, -1.36)","(22.993, 33.69, 7.665)"
1,1,A,677,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(8.772, 5.837, 3.56)","(-12.629, 12.591, -9.168)"
2,1,A,30,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(-26.86, 7.075, -0.294)","(-12.629, 12.591, -9.168)"
3,1,A,464,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(8.996, -34.672, -1.722)","(-12.629, 12.591, -9.168)"
4,1,A,590,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(11.443, 13.036, 10.453)","(-12.629, 12.591, -9.168)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,1,A,643,GLU,CA,1,A,585,GLU,CA,by-residue,25.0,upper bound,"(23.083, 20.729, -1.36)","(18.515, 1.006, 5.352)"
60,1,A,232,ASP,CA,1,A,366,GLU,CA,by-residue,25.0,upper bound,"(-6.408, -10.194, -20.008)","(-8.593, -21.735, -15.15)"
61,1,A,409,ASP,CA,1,A,319,GLU,CA,by-residue,25.0,upper bound,"(31.152, -27.044, -20.254)","(31.596, -22.01, -11.346)"
62,1,A,643,GLU,CA,1,A,319,GLU,CA,by-residue,25.0,upper bound,"(23.083, 20.729, -1.36)","(31.596, -22.01, -11.346)"


In [3]:
# Visualize the macromolecule

builder = mvs.create_builder()
structure = builder.download(url=f"https://pdb-ihm.org/cif/{cif_file}").parse(format="mmcif").assembly_structure()
core.visualize_macromolecule(structure, cif[0])

In [4]:
# Filter restraints

for filter_ in filters:
    n_before = len(restraint_df)
    restraint_df = FILTERS[filter_](restraint_df)
    print(f"The {filter_} filter removed {n_before - len(restraint_df)} rows")

if len(restraint_df) > max_restraints:
    restraint_df = core.filter_funcs.random_sample(restraint_df, n=max_restraints)

print("Restraint table post-filtering:")
restraint_df

The across_chains filter removed 57 rows
Restraint table post-filtering:


,entity_id_1,asym_id_1,seq_id_1,comp_id_1,atom_id_1,entity_id_2,asym_id_2,seq_id_2,comp_id_2,atom_id_2,model_granularity,distance_threshold,restraint_type,atom_id_1_coords,atom_id_2_coords
0,1,A,643,GLU,CA,2,B,121,LYS,CA,by-residue,25.0,upper bound,"(23.083, 20.729, -1.36)","(22.993, 33.69, 7.665)"
1,1,A,677,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(8.772, 5.837, 3.56)","(-12.629, 12.591, -9.168)"
2,1,A,30,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(-26.86, 7.075, -0.294)","(-12.629, 12.591, -9.168)"
3,1,A,464,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(8.996, -34.672, -1.722)","(-12.629, 12.591, -9.168)"
4,1,A,590,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(11.443, 13.036, 10.453)","(-12.629, 12.591, -9.168)"
5,1,A,649,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(11.773, 21.499, 6.543)","(-12.629, 12.591, -9.168)"
6,1,A,30,LYS,CA,2,B,22,LYS,CA,by-residue,35.0,upper bound,"(-26.86, 7.075, -0.294)","(-21.661, 19.264, -2.186)"


In [5]:
# Visulaize Restraints

# No special per-restraint handling
if sub_style_mode is None:
    core.visualize_restraints(structure, restraint_df)

# Requires per-restraint informatoin
else:
    SUB_STYLE_MODES[sub_style_mode](structure, restraint_df)


,entity_id_1,asym_id_1,seq_id_1,comp_id_1,atom_id_1,entity_id_2,asym_id_2,seq_id_2,comp_id_2,atom_id_2,model_granularity,distance_threshold,restraint_type,atom_id_1_coords,atom_id_2_coords,solved_distance,compliant,sub_style
3,1,A,464,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(8.996, -34.672, -1.722)","(-12.629, 12.591, -9.168)",52.505949,False,violated
0,1,A,643,GLU,CA,2,B,121,LYS,CA,by-residue,25.0,upper bound,"(23.083, 20.729, -1.36)","(22.993, 33.69, 7.665)",15.793867,True,compliant
1,1,A,677,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(8.772, 5.837, 3.56)","(-12.629, 12.591, -9.168)",25.799638,True,compliant
2,1,A,30,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(-26.86, 7.075, -0.294)","(-12.629, 12.591, -9.168)",17.654900,True,compliant
4,1,A,590,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(11.443, 13.036, 10.453)","(-12.629, 12.591, -9.168)",31.058700,True,compliant
5,1,A,649,LYS,CA,2,B,47,LYS,CA,by-residue,35.0,upper bound,"(11.773, 21.499, 6.543)","(-12.629, 12.591, -9.168)",30.358616,True,compliant
6,1,A,30,LYS,CA,2,B,22,LYS,CA,by-residue,35.0,upper bound,"(-26.86, 7.075, -0.294)","(-21.661, 19.264, -2.186)",13.385850,True,compliant


In [6]:
# Write output
builder.save_state(destination=output_file, title=f"Restraints visualized from {cif_file}")